# 🛡️ Оценка наличия сетевых атак по спектрограммам в компьютерных сетях
## Дипломын ажил — ГЛАВА 3: Эксперименталь үнэлгээ
### Dataset: CICIDS2017 + CIC-UNSW-NB15 (merged)

**Агуулга:**
- **3.3** — Базовые методы (RF, DT, LR) ба үнэлгээний шалгуур
- **3.4** — CNN архитектур (4-block Conv2D + Spectrogram)
- **3.5** — Үр дүн ба харьцуулалт
- **3.6** — Хязгаарлалт ба практик хэрэглээ

## ⚙️ 0. Суулгалт ба импорт

In [ ]:
# Шаардлагатай сангуудыг суулгах
!pip install -q tensorflow scikit-learn matplotlib seaborn pandas numpy tqdm

import os, sys, json, pickle, warnings, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, accuracy_score,
    precision_score, recall_score, f1_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

print(f"✅ TensorFlow: {tf.__version__}")
print(f"✅ GPU: {tf.config.list_physical_devices('GPU')}")
print(f"✅ Бүх сангууд амжилттай ачаалагдлаа")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## ⚙️ 1. Тохиргоо (Configuration)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  ТОХИРГОО — Судалгааны параметрүүд
#  Дипломын 3.1-т тайлбарласантай нийцүүлнэ үү
# ═══════════════════════════════════════════════════════════

# ── Google Drive замууд ────────────────────────────────
DRIVE_BASE    = "/content/drive/MyDrive"
DATASET_PATH  = os.path.join(DRIVE_BASE, "dataset_final.csv")
RESULTS_DIR   = os.path.join(DRIVE_BASE, "CNN_Results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Спектрограммын параметрүүд ─────────────────────────
N_FEATURES   = 32    # Спектрограммын өндөр (давтамжийн тэнхлэг)
WINDOW_SIZE  = 64    # Нэг спектрограммд орох урсгалын тоо (цагийн тэнхлэг)
STRIDE       = 32    # Гулгах цонхны алхам

# ── CNN сургалтын параметрүүд ──────────────────────────
BATCH_SIZE   = 64
EPOCHS       = 50
LR           = 1e-3
VAL_SIZE     = 0.15  # Баталгаажуулалтын хэсэг
TEST_SIZE    = 0.15  # Тестийн хэсэг

# ── Ангиллын горим ─────────────────────────────────────
MODE = "binary"   # "binary" (BENIGN/ATTACK) эсвэл "multiclass"

print("=" * 55)
print("  ТОХИРГОО / CONFIGURATION")
print("=" * 55)
print(f"  Dataset path : {DATASET_PATH}")
print(f"  Results dir  : {RESULTS_DIR}")
print(f"  Mode         : {MODE}")
print(f"  N_FEATURES   : {N_FEATURES}")
print(f"  WINDOW_SIZE  : {WINDOW_SIZE} урсгал")
print(f"  STRIDE       : {STRIDE}")
print(f"  EPOCHS       : {EPOCHS}")
print(f"  BATCH_SIZE   : {BATCH_SIZE}")
print(f"  LR           : {LR}")
print("=" * 55)

## 📂 2. Google Drive холбож датасет ачаалах

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("Датасет уншиж байна...")
df = pd.read_csv(DATASET_PATH, low_memory=False)
df.columns = df.columns.str.strip()
print(f"✅ Нийт: {len(df):,} мөр, {len(df.columns)} колонн")

print("\n📊 Атакийн төрлүүд (Label хуваарилалт):")
total = len(df)
for k, v in df["Label"].value_counts().items():
    bar = "█" * int(v / total * 40)
    print(f"  {k:<35}: {v:>8,}  ({v/total*100:5.2f}%)  {bar}")

bc = df["label_binary"].value_counts()
print(f"\n📊 Бинар харьцаа:")
print(f"  BENIGN (0): {bc.get(0,0):>8,}  ({bc.get(0,0)/total*100:.1f}%)")
print(f"  ATTACK (1): {bc.get(1,0):>8,}  ({bc.get(1,0)/total*100:.1f}%)")

if "source" in df.columns:
    print(f"\n📊 Эх сурвалж (Dataset source):")
    for k, v in df["source"].value_counts().items():
        print(f"  {k:<20}: {v:>8,}  ({v/total*100:.1f}%)")

## 📊 3. Анхны шинжилгээ (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("Датасетийн хуваарилалт\nCICIDS2017 + CIC-UNSW-NB15",
             fontsize=14, fontweight="bold")

# Binary
bc = df["label_binary"].value_counts()
axes[0].bar(["BENIGN (0)", "ATTACK (1)"],
            [bc.get(0,0), bc.get(1,0)],
            color=["#4CAF50", "#F44336"], edgecolor="white", linewidth=2, width=0.5)
axes[0].set_title("Бинар ангилал", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Мөрийн тоо", fontsize=11)
for i, v in enumerate([bc.get(0,0), bc.get(1,0)]):
    axes[0].text(i, v + total*0.008, f"{v:,}\n({v/total*100:.1f}%)",
                ha="center", fontsize=11, fontweight="bold")
axes[0].spines[["top","right"]].set_visible(False)
axes[0].grid(axis="y", alpha=0.3)

# Multiclass
mc = df["Label"].value_counts()
colors_mc = plt.cm.Set3(np.linspace(0, 1, len(mc)))
bars = axes[1].barh(mc.index, mc.values, color=colors_mc, edgecolor="white", linewidth=1)
axes[1].set_title("Олон классын ангилал", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Мөрийн тоо", fontsize=11)
for bar, v in zip(bars, mc.values):
    axes[1].text(v + total*0.003, bar.get_y() + bar.get_height()/2,
                f"{v:,} ({v/total*100:.1f}%)", va="center", fontsize=8)
axes[1].spines[["top","right"]].set_visible(False)
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "01_class_distribution.png"), dpi=200, bbox_inches="tight")
plt.show()
print("✅ Хадгалагдлаа: 01_class_distribution.png")

## 🔧 4. Өгөгдөл боловсруулалт (Preprocessing)

In [ ]:
# ── Тоон колоннуудыг тодорхойлох ─────────────────────────
META_COLS = ["Label", "label_binary", "source"]
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in META_COLS]
print(f"Нийт тоон колонн: {len(num_cols)}")

# ── NaN / Inf цэвэрлэх ────────────────────────────────
df_clean = df.copy()
df_clean[num_cols] = df_clean[num_cols].replace([np.inf, -np.inf], np.nan)
df_clean = df_clean.dropna(subset=num_cols).reset_index(drop=True)
print(f"NaN/Inf арилгасны дараа: {len(df_clean):,} мөр")

# ── Variance-аар шилдэг N_FEATURES-ийг сонгох ────────
variances = df_clean[num_cols].var().sort_values(ascending=False)
top_features = variances.head(N_FEATURES).index.tolist()
print(f"\nСонгосон {N_FEATURES} онцлог (variance-аар дэс дараалан):")
for i, f in enumerate(top_features):
    print(f"  {i+1:2d}. {f:<45} var={variances[f]:.4f}")

# ── MinMax масштабжуулалт [0, 1] ──────────────────────
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(df_clean[top_features].values)
print(f"\n✅ MinMax scale: {X_scaled.shape}")

# ── Шошго (Labels) ───────────────────────────────────
y_binary = df_clean["label_binary"].values.astype(np.int32)

le = LabelEncoder()
y_multi = le.fit_transform(df_clean["Label"].values)
class_names_multi = list(le.classes_)
n_classes_multi = len(class_names_multi)

if MODE == "binary":
    y          = y_binary
    n_classes  = 2
    cls_names  = ["BENIGN", "ATTACK"]
    avg_metric = "binary"
else:
    y          = y_multi
    n_classes  = n_classes_multi
    cls_names  = class_names_multi
    avg_metric = "macro"

print(f"\n✅ Mode: {MODE} | n_classes: {n_classes}")
print(f"   Класс нэрс: {cls_names}")

## 🎨 5. Спектрограмм үүсгэх (Spectrogram Generation)
*(Дипломын 3.2-т тайлбарласан арга)*

In [ ]:
def generate_spectrograms(X_flows, y_labels, window_size=WINDOW_SIZE, stride=STRIDE):
    """
    Гулгах цонхны аргаар спектрограмм үүсгэнэ.
    
    Логик:
      1. window_size урсгалыг авна → (window_size, N_FEATURES) матриц
      2. Транспон → (N_FEATURES, window_size) — спектрограмм
      3. log1p масштабжуулалт
      4. [0, 1] нормализаци
      5. Цонхны шошго = мажоритар класс
    """
    N = len(X_flows)
    n_windows = (N - window_size) // stride + 1
    n_feat = X_flows.shape[1]

    print(f"Спектрограмм үүсгэж байна...")
    print(f"  {N:,} урсгал → {n_windows:,} спектрограмм")
    print(f"  Хэмжээ: {n_feat} × {window_size} (давтамж × цаг)")

    spectrograms = np.zeros((n_windows, n_feat, window_size, 1), dtype=np.float32)
    labels_out   = np.zeros(n_windows, dtype=np.int32)

    for i in tqdm(range(n_windows), desc="Спектрограмм"):
        start = i * stride
        end   = start + window_size

        window = X_flows[start:end]      # (64, 32) — цаг × онцлог
        spec   = window.T                # (32, 64) — онцлог × цаг

        # Логарифм масштаб (STFT-тэй төстэй)
        spec = np.log1p(spec * 10.0)

        # Локал нормализаци [0, 1]
        s_min, s_max = spec.min(), spec.max()
        if s_max > s_min:
            spec = (spec - s_min) / (s_max - s_min)

        spectrograms[i, :, :, 0] = spec
        labels_out[i] = np.bincount(y_labels[start:end]).argmax()

    # Статистик
    unique, counts = np.unique(labels_out, return_counts=True)
    print(f"\n✅ Спектрограмм: {spectrograms.shape}")
    for cls_id, cnt in zip(unique, counts):
        name = cls_names[cls_id] if cls_id < len(cls_names) else str(cls_id)
        print(f"   {name:<20}: {cnt:>7,}  ({cnt/len(labels_out)*100:.1f}%)")
    return spectrograms, labels_out

spectrograms, labels = generate_spectrograms(X_scaled, y)

In [ ]:
# ── Спектрограммын жишээ зургууд ─────────────────────
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.suptitle(
    "Сетевийн трафикийн спектрограммын жишээ\n"    "(Мөр = онцлог/давтамж, Багана = цаг/урсгал)\n"    "Dataset: CICIDS2017 + CIC-UNSW-NB15",
    fontsize=13, fontweight="bold"
)

idx_0 = np.where(labels == 0)[0]
idx_1 = np.where(labels == 1)[0]

for col in range(4):
    for row, (idx_arr, title, color) in enumerate([
        (idx_0, "BENIGN (Хэвийн)", "steelblue"),
        (idx_1, "ATTACK (Халдлага)", "crimson")
    ]):
        ax = axes[row, col]
        if col < len(idx_arr):
            spec = spectrograms[idx_arr[col], :, :, 0]
            im = ax.imshow(spec, aspect="auto", origin="lower",
                          cmap="viridis", vmin=0, vmax=1)
            ax.set_title(title, color=color, fontsize=10, fontweight="bold")
            ax.set_xlabel("Цаг (урсгалууд)", fontsize=8)
            if col == 0:
                ax.set_ylabel("Онцлог (давтамж)", fontsize=8)
            ax.tick_params(labelsize=6)
        else:
            ax.axis("off")

plt.colorbar(im, ax=axes.ravel().tolist(), fraction=0.015, pad=0.04,
             label="Нормализацлагдсан утга [0,1]")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "02_spectrogram_samples.png"),
            dpi=200, bbox_inches="tight")
plt.show()
print("✅ Хадгалагдлаа: 02_spectrogram_samples.png")

## ✂️ 6. Өгөгдөл хуваах (Train / Val / Test Split)

In [ ]:
# ── CNN-д зориулсан хуваалт ────────────────────────────
X_tmp, X_test, y_tmp, y_test = train_test_split(
    spectrograms, labels,
    test_size=TEST_SIZE, random_state=SEED, stratify=labels
)
val_frac = VAL_SIZE / (1.0 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp,
    test_size=val_frac, random_state=SEED, stratify=y_tmp
)

total_sp = len(spectrograms)
print("=" * 50)
print("  ӨГӨГДӨЛ ХУВААЛТ")
print("=" * 50)
print(f"  Нийт спектрограмм : {total_sp:>8,}")
print(f"  Train (сургалт)   : {len(X_train):>8,}  ({len(X_train)/total_sp*100:.1f}%)")
print(f"  Val   (баталгаа)  : {len(X_val):>8,}  ({len(X_val)/total_sp*100:.1f}%)")
print(f"  Test  (тест)      : {len(X_test):>8,}  ({len(X_test)/total_sp*100:.1f}%)")
print("=" * 50)

# ── Класс жин (дисбаланс тэнцвэржүүлэлт) ────────────
classes_arr = np.unique(y_train)
weights_arr = compute_class_weight("balanced", classes=classes_arr, y=y_train)
class_weight_dict = dict(zip(classes_arr.astype(int), weights_arr))
print(f"\n  Класс жин: {class_weight_dict}")

# ── Baseline-д зориулсан хавтгай онцлог ───────────────
# Спектрограмм бүрийн дундаж (mean pooling) → flat vector
X_flat = spectrograms.mean(axis=(2, 3))   # (N, N_FEATURES)
X_flat_tmp, X_flat_test, yf_tmp, yf_test = train_test_split(
    X_flat, labels, test_size=TEST_SIZE, random_state=SEED, stratify=labels
)
X_flat_train, X_flat_val, yf_train, yf_val = train_test_split(
    X_flat_tmp, yf_tmp, test_size=val_frac, random_state=SEED, stratify=yf_tmp
)
print(f"\n  Baseline flat features: {X_flat_train.shape}")

## 📊 3.3 Базовые методы ба үнэлгээний шалгуур
*(Глава 3.3 — Базовые методы и критерии оценки эффективности)*

In [ ]:
# ══════════════════════════════════════════════════════════
#  ГЛАВА 3.3 — БАЗОВЫЕ МЕТОДЫ (Baseline Methods)
#  Random Forest, Decision Tree, Logistic Regression
# ══════════════════════════════════════════════════════════

baseline_results = {}

# ── 1. Random Forest ───────────────────────────────────
print("[1/3] Random Forest сургаж байна...")
t0 = time.time()
rf = RandomForestClassifier(n_estimators=200, max_depth=20,
                             random_state=SEED, n_jobs=-1, class_weight="balanced")
rf.fit(X_flat_train, yf_train)
rf_pred = rf.predict(X_flat_test)
t_rf = time.time() - t0
baseline_results["Random Forest"] = {
    "accuracy" : accuracy_score(yf_test, rf_pred),
    "precision": precision_score(yf_test, rf_pred, average=avg_metric, zero_division=0),
    "recall"   : recall_score(yf_test, rf_pred, average=avg_metric, zero_division=0),
    "f1"       : f1_score(yf_test, rf_pred, average=avg_metric, zero_division=0),
    "time"     : t_rf
}
print(f"  ✅ RF  → Accuracy={baseline_results['Random Forest']['accuracy']:.4f}, "      f"F1={baseline_results['Random Forest']['f1']:.4f}, Хугацаа={t_rf:.1f}с")

# ── 2. Decision Tree ───────────────────────────────────
print("[2/3] Decision Tree сургаж байна...")
t0 = time.time()
dt = DecisionTreeClassifier(max_depth=20, random_state=SEED, class_weight="balanced")
dt.fit(X_flat_train, yf_train)
dt_pred = dt.predict(X_flat_test)
t_dt = time.time() - t0
baseline_results["Decision Tree"] = {
    "accuracy" : accuracy_score(yf_test, dt_pred),
    "precision": precision_score(yf_test, dt_pred, average=avg_metric, zero_division=0),
    "recall"   : recall_score(yf_test, dt_pred, average=avg_metric, zero_division=0),
    "f1"       : f1_score(yf_test, dt_pred, average=avg_metric, zero_division=0),
    "time"     : t_dt
}
print(f"  ✅ DT  → Accuracy={baseline_results['Decision Tree']['accuracy']:.4f}, "      f"F1={baseline_results['Decision Tree']['f1']:.4f}, Хугацаа={t_dt:.1f}с")

# ── 3. Logistic Regression ────────────────────────────
print("[3/3] Logistic Regression сургаж байна...")
t0 = time.time()
lr_clf = LogisticRegression(max_iter=2000, random_state=SEED,
                             n_jobs=-1, class_weight="balanced")
lr_clf.fit(X_flat_train, yf_train)
lr_pred = lr_clf.predict(X_flat_test)
t_lr = time.time() - t0
baseline_results["Logistic Regression"] = {
    "accuracy" : accuracy_score(yf_test, lr_pred),
    "precision": precision_score(yf_test, lr_pred, average=avg_metric, zero_division=0),
    "recall"   : recall_score(yf_test, lr_pred, average=avg_metric, zero_division=0),
    "f1"       : f1_score(yf_test, lr_pred, average=avg_metric, zero_division=0),
    "time"     : t_lr
}
print(f"  ✅ LR  → Accuracy={baseline_results['Logistic Regression']['accuracy']:.4f}, "      f"F1={baseline_results['Logistic Regression']['f1']:.4f}, Хугацаа={t_lr:.1f}с")

print("\n" + "="*70)
print(f"  {'Арга':<22} {'Acc':>8} {'Prec':>8} {'Recall':>8} {'F1':>8} {'Хугацаа':>10}")
print("-"*70)
for name, m in baseline_results.items():
    print(f"  {name:<22} {m['accuracy']:>8.4f} {m['precision']:>8.4f} "          f"{m['recall']:>8.4f} {m['f1']:>8.4f} {m['time']:>9.1f}с")
print("="*70)

## 🧠 3.4 CNN Архитектур
*(Глава 3.4 — Архитектура CNN и реализация)*

In [ ]:
# ══════════════════════════════════════════════════════════
#  ГЛАВА 3.4 — CNN АРХИТЕКТУР
#  4 Conv блок → GlobalAveragePooling → Dense → Output
#  Оролт: (32, 64, 1) спектрограмм зураг
# ══════════════════════════════════════════════════════════

def build_cnn(input_shape, n_classes):
    """
    4-блок CNN архитектур спектрограмм дүн шинжилгээнд.

    Блок бүр: Conv2D → BatchNorm → MaxPool → Dropout
    Сүүлд  : GlobalAveragePooling → Dense(256) → Dense(128) → Output

    Параметрүүд:
        input_shape : (H, W, C) = (32, 64, 1)
        n_classes   : 2 (binary) эсвэл N (multiclass)
    """
    inp = keras.Input(shape=input_shape, name="spectrogram_input")

    # ── Блок 1 — 32 шүүлтүүр ──────────────────────────
    x = layers.Conv2D(32, (3,3), padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(1e-4), name="conv1")(inp)
    x = layers.BatchNormalization(name="bn1")(x)
    x = layers.MaxPooling2D((2,2), name="pool1")(x)
    x = layers.Dropout(0.25, name="drop1")(x)

    # ── Блок 2 — 64 шүүлтүүр ──────────────────────────
    x = layers.Conv2D(64, (3,3), padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(1e-4), name="conv2")(x)
    x = layers.BatchNormalization(name="bn2")(x)
    x = layers.MaxPooling2D((2,2), name="pool2")(x)
    x = layers.Dropout(0.25, name="drop2")(x)

    # ── Блок 3 — 128 шүүлтүүр ─────────────────────────
    x = layers.Conv2D(128, (3,3), padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(1e-4), name="conv3")(x)
    x = layers.BatchNormalization(name="bn3")(x)
    x = layers.MaxPooling2D((2,2), name="pool3")(x)
    x = layers.Dropout(0.30, name="drop3")(x)

    # ── Блок 4 — 256 шүүлтүүр ─────────────────────────
    x = layers.Conv2D(256, (3,3), padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(1e-4), name="conv4")(x)
    x = layers.BatchNormalization(name="bn4")(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.40, name="drop4")(x)

    # ── Бүтэн холбоос давхаргууд ───────────────────────
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=regularizers.l2(1e-4), name="fc1")(x)
    x = layers.Dropout(0.40, name="drop5")(x)
    x = layers.Dense(128, activation="relu",
                     kernel_regularizer=regularizers.l2(1e-4), name="fc2")(x)
    x = layers.Dropout(0.30, name="drop6")(x)

    # ── Гарлын давхарга ───────────────────────────────
    if n_classes == 2:
        out = layers.Dense(1, activation="sigmoid", name="output")(x)
    else:
        out = layers.Dense(n_classes, activation="softmax", name="output")(x)

    return keras.Model(inputs=inp, outputs=out, name="CNN_IDS_Spectrogram")

# Модель үүсгэх
input_shape = (N_FEATURES, WINDOW_SIZE, 1)
model = build_cnn(input_shape, n_classes)

# Compile
optimizer = keras.optimizers.Adam(learning_rate=LR)
if n_classes == 2:
    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy",
                 keras.metrics.AUC(name="auc"),
                 keras.metrics.Precision(name="precision"),
                 keras.metrics.Recall(name="recall")]
    )
else:
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy",
                 keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_acc")]
    )

model.summary()
print(f"\n✅ Нийт параметр: {model.count_params():,}")
print(f"   Оролтын хэмжээ: {input_shape}")
print(f"   Гарлын ангиллын тоо: {n_classes}")

## 🚀 7. CNN Сургах (Training)

In [ ]:
model_path = os.path.join(RESULTS_DIR, "cnn_ids_best.keras")

cbs = [
    keras.callbacks.ModelCheckpoint(
        model_path, monitor="val_accuracy",
        save_best_only=True, verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=10,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5,
        patience=5, min_lr=1e-6, verbose=1
    )
]

print(f"CNN сургаж байна: max {EPOCHS} эпох, batch={BATCH_SIZE}, lr={LR}")
print(f"Модел хадгалах: {model_path}")
print("-" * 55)

t_start = time.time()
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=cbs,
    verbose=1
)
train_time = time.time() - t_start

best_val_acc = max(history.history["val_accuracy"])
best_epoch   = history.history["val_accuracy"].index(best_val_acc) + 1
print(f"\n✅ Сургалт дууслаа!")
print(f"   Хугацаа        : {train_time/60:.1f} минут")
print(f"   Нийт эпох      : {len(history.history['accuracy'])}")
print(f"   Шилдэг val_acc : {best_val_acc:.4f}  (эпох {best_epoch})")

In [ ]:
# ── Сургалтын муруй зурах ────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    "CNN Сургалтын үр дүн — Spectrogram IDS\n"    "(CICIDS2017 + CIC-UNSW-NB15, Binary Classification)",
    fontsize=14, fontweight="bold"
)

ep = range(1, len(history.history["accuracy"]) + 1)

# Accuracy
ax1.plot(ep, history.history["accuracy"],  color="#2196F3", lw=2.5, label="Train")
ax1.plot(ep, history.history["val_accuracy"], color="#4CAF50", lw=2.5, ls="--", label="Validation")
ax1.set_title("Нарийвчлал (Accuracy)", fontsize=12)
ax1.set_xlabel("Эпох", fontsize=11); ax1.set_ylabel("Accuracy", fontsize=11)
ax1.legend(fontsize=10); ax1.grid(alpha=0.3); ax1.set_ylim(0, 1.05)
ax1.annotate(f"Шилдэг: {best_val_acc:.4f}\n(эпох {best_epoch})",
             xy=(best_epoch, best_val_acc),
             xytext=(best_epoch+2, best_val_acc-0.08),
             arrowprops=dict(arrowstyle="->", color="gray"),
             fontsize=9, color="darkgreen")

# Loss
ax2.plot(ep, history.history["loss"],     color="#F44336", lw=2.5, label="Train")
ax2.plot(ep, history.history["val_loss"], color="#FF9800", lw=2.5, ls="--", label="Validation")
ax2.set_title("Алдагдал (Loss)", fontsize=12)
ax2.set_xlabel("Эпох", fontsize=11); ax2.set_ylabel("Loss", fontsize=11)
ax2.legend(fontsize=10); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "03_training_curves.png"), dpi=200, bbox_inches="tight")
plt.show()
print("✅ Хадгалагдлаа: 03_training_curves.png")

## 📈 3.5 Үр дүн ба харьцуулал
*(Глава 3.5 — Результаты классификации и сравнительный анализ)*

In [ ]:
# ══════════════════════════════════════════════════════════
#  ГЛАВА 3.5 — CNN ҮНЭЛГЭЭ
# ══════════════════════════════════════════════════════════

y_prob = model.predict(X_test, batch_size=64, verbose=0)
if n_classes == 2:
    y_pred    = (y_prob.squeeze() >= 0.5).astype(int)
    y_prob_1d = y_prob.squeeze()
else:
    y_pred = np.argmax(y_prob, axis=1)

cnn_metrics = {
    "accuracy" : accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, average=avg_metric, zero_division=0),
    "recall"   : recall_score(y_test, y_pred, average=avg_metric, zero_division=0),
    "f1"       : f1_score(y_test, y_pred, average=avg_metric, zero_division=0),
    "time"     : train_time
}

print("=" * 55)
print("  CNN — ТЕСТ ХЭСГИЙН ҮНЭЛГЭЭ")
print("=" * 55)
print(f"  Accuracy  : {cnn_metrics['accuracy']:.4f}  ({cnn_metrics['accuracy']*100:.2f}%)")
print(f"  Precision : {cnn_metrics['precision']:.4f}")
print(f"  Recall    : {cnn_metrics['recall']:.4f}")
print(f"  F1-Score  : {cnn_metrics['f1']:.4f}")

# ── Нарийвчилсан тайлан ────────────────────────────────
print("\n  Нарийвчилсан тайлан (Classification Report):")
print(classification_report(y_test, y_pred, target_names=cls_names,
                            digits=4, zero_division=0))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

n = len(cls_names)
fig_size = max(8, n * 1.4)
fig, ax = plt.subplots(figsize=(fig_size, fig_size * 0.85))

im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(cls_names, rotation=45, ha="right", fontsize=10)
ax.set_yticklabels(cls_names, fontsize=10)
ax.set_xlabel("Таамагласан класс", fontsize=12)
ax.set_ylabel("Бодит класс", fontsize=12)
ax.set_title(
    "Confusion Matrix — CNN Spectrogram IDS\n"    "CICIDS2017 + CIC-UNSW-NB15",
    fontsize=13, fontweight="bold"
)

for i in range(n):
    for j in range(n):
        val   = cm_norm[i, j]
        count = cm[i, j]
        color = "white" if val > 0.55 else "black"
        ax.text(j, i, f"{val:.3f}\n({count:,})",
               ha="center", va="center", fontsize=9,
               color=color, fontweight="bold" if i==j else "normal")

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "04_confusion_matrix.png"), dpi=200, bbox_inches="tight")
plt.show()
print("✅ Хадгалагдлаа: 04_confusion_matrix.png")

In [ ]:
# ── ROC Кривая (Binary) ───────────────────────────────
if n_classes == 2:
    fpr, tpr, _ = roc_curve(y_test, y_prob_1d)
    roc_auc = auc(fpr, tpr)
    cnn_metrics["auc"] = roc_auc

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.plot(fpr, tpr, color="#2196F3", lw=2.5,
            label=f"CNN Spectrogram (AUC = {roc_auc:.4f})")
    ax.plot([0,1],[0,1], color="gray", ls="--", lw=1.5,
            label="Санамсаргүй (AUC = 0.50)")
    ax.fill_between(fpr, tpr, alpha=0.1, color="#2196F3")
    ax.set_xlabel("False Positive Rate (FPR)", fontsize=12)
    ax.set_ylabel("True Positive Rate (TPR)", fontsize=12)
    ax.set_title(
        "ROC Кривая — CNN Spectrogram IDS\n"        "CICIDS2017 + CIC-UNSW-NB15",
        fontsize=13, fontweight="bold"
    )
    ax.legend(fontsize=11, loc="lower right")
    ax.grid(alpha=0.3)
    ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.05)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "05_roc_curve.png"), dpi=200, bbox_inches="tight")
    plt.show()
    print(f"✅ AUC-ROC: {roc_auc:.4f}")
    print("✅ Хадгалагдлаа: 05_roc_curve.png")

In [ ]:
# ── Метрикийн гистограмм ──────────────────────────────
keys_bar   = ["accuracy", "precision", "recall", "f1"]
lbl_bar    = ["Accuracy", "Precision", "Recall", "F1-Score"]
vals_bar   = [cnn_metrics[k] for k in keys_bar]
clrs_bar   = ["#4CAF50", "#2196F3", "#FF9800", "#9C27B0"]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(lbl_bar, vals_bar, color=clrs_bar, edgecolor="white", lw=2, width=0.5)
for bar, val in zip(bars, vals_bar):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.012,
            f"{val:.4f}", ha="center", va="bottom", fontsize=13, fontweight="bold")
ax.set_ylim(0, 1.18)
ax.set_ylabel("Үнэлгээний утга", fontsize=12)
ax.set_title(
    "CNN Spectrogram IDS — Тест хэсгийн метрик\n"    "CICIDS2017 + CIC-UNSW-NB15",
    fontsize=13, fontweight="bold"
)
ax.grid(axis="y", alpha=0.3)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "06_metrics_bar.png"), dpi=200, bbox_inches="tight")
plt.show()
print("✅ Хадгалагдлаа: 06_metrics_bar.png")

## 📊 3.5 Аргуудын харьцуулалт (Comparison Table)

In [ ]:
# ══════════════════════════════════════════════════════════
#  ГЛАВА 3.5 — ХАРЬЦУУЛАЛТЫН ХҮСНЭГТ
# ══════════════════════════════════════════════════════════

all_results = {
    **baseline_results,
    "CNN (Spectrogram)": cnn_metrics
}

print("\n" + "="*78)
print("  ГЛАВА 3.5 — АРГУУДЫН ХАРЬЦУУЛАЛТ")
print("  Dataset: CICIDS2017 + CIC-UNSW-NB15 | Mode:", MODE)
print("="*78)
print(f"  {'Арга':<25} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1-Score':>10}")
print("-"*78)

best_f1 = max(m["f1"] for m in all_results.values())
for name, m in all_results.items():
    marker = "  ◄ ШИЛДЭГ" if m["f1"] == best_f1 else ""
    print(f"  {name:<25} {m['accuracy']:>10.4f} {m['precision']:>10.4f} "          f"{m['recall']:>10.4f} {m['f1']:>10.4f}{marker}")
print("="*78)

if "auc" in cnn_metrics:
    print(f"\n  CNN AUC-ROC: {cnn_metrics['auc']:.4f}")
print(f"  CNN сургалтын хугацаа: {train_time/60:.1f} минут")

In [ ]:
# ── Харьцуулалтын гистограмм ──────────────────────────
methods = list(all_results.keys())
metrics_cmp = ["accuracy", "precision", "recall", "f1"]
labels_cmp  = ["Accuracy", "Precision", "Recall", "F1-Score"]
colors_cmp  = ["#4CAF50", "#2196F3", "#FF9800", "#9C27B0"]

x = np.arange(len(methods))
width = 0.18

fig, ax = plt.subplots(figsize=(14, 6))
for i, (metric, lbl, color) in enumerate(zip(metrics_cmp, labels_cmp, colors_cmp)):
    vals = [all_results[m][metric] for m in methods]
    ax.bar(x + i*width - 1.5*width, vals, width, label=lbl, color=color, alpha=0.88, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=15, ha="right", fontsize=11)
ax.set_ylim(0, 1.18)
ax.set_ylabel("Үнэлгээний утга", fontsize=12)
ax.set_title(
    "Аргуудын харьцуулалт — Сетевых атак илрүүлэлт\n"    "CICIDS2017 + CIC-UNSW-NB15 (Spectrogram-based CNN vs Baseline)",
    fontsize=13, fontweight="bold"
)
ax.legend(fontsize=10, loc="lower right")
ax.grid(axis="y", alpha=0.3)
ax.spines[["top","right"]].set_visible(False)

# CNN баганыг тодруулах
cnn_idx = list(methods).index("CNN (Spectrogram)")
ax.axvspan(cnn_idx - 0.4, cnn_idx + 0.4, alpha=0.07, color="purple", zorder=0)
ax.text(cnn_idx, 1.12, "CNN", ha="center", fontsize=10, color="purple", fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "07_comparison_chart.png"), dpi=200, bbox_inches="tight")
plt.show()
print("✅ Хадгалагдлаа: 07_comparison_chart.png")

## 💾 8. Бүх үр дүнг хадгалах

In [ ]:
# ── JSON метрик ──────────────────────────────────────
export = {}
for name, m in all_results.items():
    export[name] = {k: float(v) for k, v in m.items() if isinstance(v, (int, float, np.floating))}

with open(os.path.join(RESULTS_DIR, "all_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(export, f, indent=2, ensure_ascii=False)

# ── Дипломын хүснэгт хэвлэх (LaTeX/Word-д) ──────────
print("\n📋 ДИПЛОМЫН ХҮСНЭГТ (Глава 3.5):")
print("-" * 70)
print(f"{'Арга':<25} | {'Acc':>8} | {'Prec':>8} | {'Recall':>8} | {'F1':>8}")
print("-" * 70)
for name, m in all_results.items():
    print(f"{name:<25} | {m['accuracy']:>8.4f} | {m['precision']:>8.4f} | {m['recall']:>8.4f} | {m['f1']:>8.4f}")
print("-" * 70)

# ── Хадгалсан файлуудын жагсаалт ─────────────────────
print("\n📁 DRIVE-Д ХАДГАЛСАН ФАЙЛУУД:")
print(f"   {RESULTS_DIR}")
print("-" * 55)
for fname in sorted(os.listdir(RESULTS_DIR)):
    fpath = os.path.join(RESULTS_DIR, fname)
    size  = os.path.getsize(fpath) / 1024
    print(f"   {fname:<45} {size:>7.1f} KB")

print("\n✅ БҮГД ХАДГАЛАГДЛАА!")
print("\n🎓 Дипломын 3-р бүлэгт ашиглах зургууд:")
figs = [
    ("01_class_distribution.png",  "3.2 — Датасетийн хуваарилалт"),
    ("02_spectrogram_samples.png", "3.2 — Спектрограммын жишээ"),
    ("03_training_curves.png",     "3.4 — Сургалтын муруй"),
    ("04_confusion_matrix.png",    "3.5 — Confusion Matrix"),
    ("05_roc_curve.png",           "3.5 — ROC кривая"),
    ("06_metrics_bar.png",         "3.5 — Метрикийн гистограмм"),
    ("07_comparison_chart.png",    "3.5 — Харьцуулалтын гистограмм"),
]
for fname, chapter in figs:
    print(f"   📊 {fname:<40} → {chapter}")